# 🚀 Gated Recurrent Unit (GRU) — Solutions Notebook

**This notebook contains complete, verified solutions.**

**Difficulty**: ⭐⭐ Intermediate  
**Time**: ~45 minutes

---


## 🎯 Section 1: Overview

The **Gated Recurrent Unit (GRU)** is a simplified gating mechanism variant of the LSTM. It has fewer parameters because it merges the cell state and hidden state, and combines the forget and input gates into a single **update gate**.

### Why use GRU?
- Fewer parameters -> less prone to overfitting
- Faster training speed


## 📐 Section 2: Math & Intuition

### GRU Cell Equations
Given input $x_t$ and previous hidden state $h_{t-1}$:
1. **Reset Gate**: $r_t = \sigma(x_t W_{xr} + h_{t-1} W_{hr} + b_r)$
2. **Update Gate**: $z_t = \sigma(x_t W_{xz} + h_{t-1} W_{hz} + b_z)$
3. **Candidate Hidden State**: $\tilde{h}_t = \tanh(x_t W_{xh} + (r_t * h_{t-1}) W_{hh} + b_h)$
4. **Hidden State Update**: $h_t = (1 - z_t) * h_{t-1} + z_t * \tilde{h}_t$

The reset gate determines how to combine new input with past memory, while the update gate decides how much of the past state to keep.


## 🔧 Section 3: Implementation from Scratch


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('GRU Setup complete! ✅')


### 3.1 GRU Cell Forward Pass


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def gru_cell_forward(xt, h_prev, Wx, Wh, bx, bh):
    """
    xt: input at step t of shape (batch_size, input_dim)
    h_prev: previous hidden state of shape (batch_size, hidden_dim)
    Wx: input weights for [r, z, h] of shape (input_dim, 3 * hidden_dim)
    Wh: hidden weights for [r, z, h] of shape (hidden_dim, 3 * hidden_dim)
    bx: input biases of shape (1, 3 * hidden_dim)
    bh: hidden biases of shape (1, 3 * hidden_dim)
    """
    hidden_dim = h_prev.shape[1]
    
    # Project inputs and hidden states
    X_proj = xt @ Wx + bx
    H_proj = h_prev @ Wh + bh
    
    # Split projects
    xr, xz, xh = np.split(X_proj, 3, axis=1)
    hr, hz, hh = np.split(H_proj, 3, axis=1)
    
    r = sigmoid(xr + hr)
    z = sigmoid(xz + hz)
    
    # Candidate hidden state uses reset gate to mask past state
    h_cand = np.tanh(xh + r * hh)
    
    # Update hidden state
    h_next = (1 - z) * h_prev + z * h_cand
    
    return h_next


### 3.2 Verify GRU Cell


In [ ]:
xt_v = np.random.randn(2, 3)
h_v = np.zeros((2, 4))
Wx_v = np.random.randn(3, 12)
Wh_v = np.random.randn(4, 12)
bx_v = np.zeros((1, 12))
bh_v = np.zeros((1, 12))

h_n = gru_cell_forward(xt_v, h_v, Wx_v, Wh_v, bx_v, bh_v)
print('GRU state shape (should be [2, 4]):', list(h_n.shape))
if 'TODO' not in gru_cell_forward.__code__.co_consts:
    assert list(h_n.shape) == [2, 4]
    print('GRU forward pass verified! ✅')


## 📦 Section 4: Library Implementation


In [ ]:
import torch
import torch.nn as nn

class PyTorchGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        out, h_n = self.gru(x)
        return self.fc(out[:, -1, :])
        


In [ ]:
model = PyTorchGRU(input_dim=4, hidden_dim=8, output_dim=1)
dummy_batch = torch.randn(5, 10, 4)
out = model(dummy_batch)
print('Output shape (should be [5, 1]):', list(out.shape))
assert list(out.shape) == [5, 1]
print('PyTorch GRU check passed! ✅')


## 🧪 Section 5: Experiments


Compare the parameter efficiency of RNN, LSTM, and GRU.


In [ ]:
input_dim = 10
hidden_dim = 20
output_dim = 2

rnn = nn.RNN(input_dim, hidden_dim, batch_first=True)
lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
gru = nn.GRU(input_dim, hidden_dim, batch_first=True)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

print('Parameter Counts:')
print('Vanilla RNN:', count_params(rnn))
print('LSTM:       ', count_params(lstm))
print('GRU:        ', count_params(gru))

assert count_params(gru) < count_params(lstm)
print('Verification: GRU has fewer parameters than LSTM! ✅')


## ❓ Section 6: Interview Questions


### Q1: How does a GRU differ from an LSTM in terms of gates and states?
**Answer**:
- **States**: LSTM maintains two states: hidden state $h_t$ and cell state $C_t$. GRU maintains only a single hidden state $h_t$.
- **Gates**: LSTM has $3$ gates (forget, input, output). GRU has only $2$ gates (reset, update). This reduction makes GRU computationally lighter.

### Q2: Why might you choose a GRU over an LSTM?
**Answer**:
GRU models have fewer weights, making them faster to compute per iteration and less likely to overfit on smaller sequential datasets. If training latency or storage size is a constraint, GRUs often perform comparably to LSTMs with reduced overhead.

### Q3: Explain the function of the Reset Gate in GRU.
**Answer**:
The Reset Gate $r_t$ determines how much of the past hidden state $h_{t-1}$ to write into the candidate hidden state calculation $\tilde{h}_t$. If $r_t \approx 0.0$, the candidate drops the historical hidden state entirely and acts as if processing a new sequence, which is useful for segments that change context suddenly.


## 🏆 Section 7: Challenge — Bidirectional GRU


**Challenge**: Implement a Bidirectional GRU layer wrapper using PyTorch's native `nn.GRU` layer.


In [ ]:
class BiGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)  # * 2 because output is concat of forward & backward
        
    def forward(self, x):
        out, h_n = self.gru(x)
        # Take the output of the last sequence step (forward + backward concats)
        last_step = out[:, -1, :]
        return self.fc(last_step)

model_bi = BiGRU(5, 8, 2)
dummy_batch = torch.randn(4, 12, 5)
out_bi = model_bi(dummy_batch)
print('BiGRU output shape (should be [4, 2]):', list(out_bi.shape))
if 'TODO' not in BiGRU.__init__.__code__.co_consts:
    assert list(out_bi.shape) == [4, 2]
    print('BiGRU wrapper verified! ✅')
